<a href="https://colab.research.google.com/github/yuichi-dev-creator/kaggle-baseline/blob/main/notebooks/train_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!git clone https://github.com/yuichi-dev-creator/kaggle-baseline.git

fatal: destination path 'kaggle-baseline' already exists and is not an empty directory.


In [12]:
%cd kaggle-baseline

[Errno 20] Not a directory: 'kaggle-baseline'
/content/kaggle-baseline


In [13]:
!ls

kaggle-baseline  notebooks  src


In [14]:
import pandas as pd
from src.preprocess import preprocess

In [15]:
df = pd.DataFrame({
    "Rainfall_mm": [10, 20],
    "Previous_Irrigation_mm": [5, 3]
})

df = preprocess(df)
df

,Rainfall_mm,Previous_Irrigation_mm,water_balance
0,10,5,5
1,20,3,17


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

from src.preprocess import preprocess

In [17]:
df = pd.DataFrame({
    "Rainfall_mm": [10, 20, 15, 30],
    "Previous_Irrigation_mm": [5, 3, 7, 10],
    "target": [100, 200, 150, 300]
})

df = preprocess(df)

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = lgb.LGBMRegressor()
model.fit(X_train, y_train)

pred = model.predict(X_valid)

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 3, number of used features: 0
[LightGBM] [Info] Start training from score 183.333333
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the spl

In [18]:
rmse = mean_squared_error(y_valid, pred) ** 0.5
print("RMSE:", rmse)

RMSE: 16.666666666666657


In [19]:
import pandas as pd

train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

print(train.shape)
print(test.shape)
train.head()

(630000, 21)
(270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [20]:
train.columns

Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='object')

In [21]:
from src.preprocess import preprocess

train = preprocess(train)
test = preprocess(test)

In [22]:
X = train.drop("Irrigation_Need", axis=1)
y = train["Irrigation_Need"]

In [23]:
cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    X[col] = X[col].astype("category")
    test[col] = test[col].astype("category")

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
import lightgbm as lgb

model = lgb.LGBMClassifier()

model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034597 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3216
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 21
[LightGBM] [Info] Start training from score -3.403581
[LightGBM] [Info] Start training from score -0.531609
[LightGBM] [Info] Start training from score -0.969989


LGBMClassifier()

In [26]:
test_pred = model.predict(test)

In [34]:
import pandas as pd

sub = pd.DataFrame({
    "id": test["id"],
    "Irrigation_Need": test_pred
})

sub.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [35]:
sub.to_csv("/content/submission.csv", index=False)

In [36]:
!ls

kaggle-baseline  notebooks  src  submission.csv
